In [1]:
import glob
import cv2
import numpy as np
import matplotlib.pyplot as plt
import json

In [2]:
def calibrate_camera_from_checkerboard(
    images_glob: str,
    pattern_size=(9, 6),
    square_size_m=0.025,
    visualize=False,
    K=None,
    fisheye=False,
):
    nx, ny = pattern_size

    objp = np.zeros((nx * ny, 3), np.float32)
    objp[:, :2] = np.mgrid[0:nx, 0:ny].T.reshape(-1, 2)
    objp *= float(square_size_m)

    objpoints = []
    imgpoints = []

    images = sorted(glob.glob(images_glob))
    if not images:
        raise FileNotFoundError(f"No images found for glob: {images_glob}")

    print(f"Total number of available images = {len(images)}")

    criteria = (
        cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER,
        40,
        1e-6,
    )

    image_size = None

    for fname in images:
        img = cv2.imread(fname)
        if img is None:
            continue

        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

        if image_size is None:
            image_size = (gray.shape[1], gray.shape[0])
            print(f"image size = {image_size}")

        found, corners = cv2.findChessboardCorners(
            gray,
            (nx, ny),
            flags=(
                cv2.CALIB_CB_ADAPTIVE_THRESH
                + cv2.CALIB_CB_NORMALIZE_IMAGE
                + cv2.CALIB_CB_ACCURACY
            ),
        )

        if not found:
            print(f"[WARN] No corners found: {fname}")
            continue

        corners_refined = cv2.cornerSubPix(
            gray,
            corners,
            winSize=(11, 11),
            zeroZone=(-1, -1),
            criteria=criteria,
        )

        if fisheye:
            objpoints.append(objp.reshape(1, -1, 3).astype(np.float64))
            imgpoints.append(corners_refined.reshape(1, -1, 2).astype(np.float64))
        else:
            objpoints.append(objp.astype(np.float32))
            imgpoints.append(corners_refined.astype(np.float32))

        if visualize:
            vis = img.copy()
            cv2.drawChessboardCorners(vis, (nx, ny), corners_refined, found)
            vis_rgb = cv2.cvtColor(vis, cv2.COLOR_BGR2RGB)
            plt.imshow(vis_rgb)
            plt.show()

    if len(objpoints) < 10:
        raise RuntimeError(f"Too few valid images ({len(objpoints)}). Aim for 15–30+.")

    print(f"Images with corners = {len(objpoints)}")

    if fisheye:
        K_init = np.zeros((3, 3), dtype=np.float64) if K is None else K.astype(np.float64)
        D_init = np.zeros((4, 1), dtype=np.float64)

        cal_flags = (
            cv2.fisheye.CALIB_RECOMPUTE_EXTRINSIC
            + cv2.fisheye.CALIB_CHECK_COND
            + cv2.fisheye.CALIB_FIX_SKEW
        )

        if K is not None:
            cal_flags += cv2.fisheye.CALIB_USE_INTRINSIC_GUESS

        rms, K_out, dist, rvecs, tvecs = cv2.fisheye.calibrate(
            objpoints,
            imgpoints,
            image_size,
            K_init,
            D_init,
            flags=cal_flags,
            criteria=criteria,
        )

    else:
        cal_flags = cv2.CALIB_FIX_K3

        if K is not None:
            cal_flags += cv2.CALIB_USE_INTRINSIC_GUESS

        rms, K_out, dist, rvecs, tvecs = cv2.calibrateCamera(
            objpoints,
            imgpoints,
            image_size,
            K,
            None,
            flags=cal_flags,
        )

    return image_size[0], image_size[1], K_out, dist, rvecs, tvecs, rms

In [8]:
def save_calibration(H, W,
                     K, dist,  RMS, 
                     calibration_file, 
                     fisheye=False):
    fx = K[0, 0]
    cx = K[0, 2]
    fy = K[1, 1]
    cy = K[1, 2]

    k1 = dist[0, 0]
    k2 = dist[0, 1]

    if fisheye:
        camera_type = "OPENCV_FISHEYE"
    else:
        camera_type = "OPENCV"

    cam_calib = {"camera_type":camera_type, 
                 "h":H, 
                 "w":W, 
                 "fx":fx, 
                 "fy":fy, 
                 "cx":cx, 
                 "cy":cy, 
                 "k1":k1, 
                 "k2":k2}

    if fisheye:
        k3 = dist[0, 2]
        k4 = dist[0, 3]

        cam_calib["k3"] = k3
        cam_calib["k4"] = k4
    else:
        k3 = dist[0, 4]
        p1 = dist[0, 2]
        p2 = dist[0, 3]

        cam_calib["k3"] = k3
        cam_calib["p1"] = p1
        cam_calib["p2"] = p2

    cam_calib["RMS"] = RMS

    with open(calibration_file, 'w') as f:
        json.dump(cam_calib, f, indent=4)

In [11]:
calibration_images = "/data/datasets//calibration/16_9_wfov_images/*.JPG"
is_fisheye=True
cal_file = "/data/datasets/calibration/cal_val_images_16_9_wfov_fisheye.json"

W, H, K, dist, rvecs, tvecs, rms = calibrate_camera_from_checkerboard(
        images_glob=calibration_images,
        pattern_size=(9, 6),
        square_size_m=0.025,  # set to your printed square size
        visualize=True,
        K = None,
        fisheye=is_fisheye
    )

print("RMS reprojection error:", rms)
print("K (intrinsics):\n", K)
print("dist (distortion):\n", dist)

In [10]:
save_calibration(H, W, K, dist, rms, cal_file, is_fisheye)